# ML-08 — Capstone Modeling Lane

## 1. Method choice
We compare Logistic Regression, a shallow Decision Tree, and Random Forest. Random Forest is selected by **Precision@50**, matching the operational goal of a high-quality short review queue.

## 2. Split design
Use a **client holdout**: approximately 20% of clients are held out so pages from the same client do not appear in both train and test. This is more honest than a row-random split when client-specific patterns exist.

In [ ]:
import json
from pathlib import Path
ROOT=Path.cwd()
while ROOT.name and not (ROOT/'outputs'/'model_results.json').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
r=json.loads((ROOT/'outputs'/'model_results.json').read_text())
rows=[]
for name,m in r['models'].items(): rows.append({'model':name,**{k:m[k] for k in ['roc_auc','average_precision','precision_at_50','recall','f1']}})
rows.append({'model':'baseline_rules',**{k:r['baseline'][k] for k in ['baseline_roc_auc','baseline_average_precision','baseline_precision_at_50','baseline_recall','baseline_f1']}})
import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))

## 3. Result
On the held-out client test set, Random Forest achieved ROC-AUC **0.750**, average precision **0.618**, and Precision@50 **0.740**. The baseline achieved Precision@50 **0.240** and ROC-AUC **0.627**.

## 4. Interpretation
The model is useful for prioritization, not automatic action. The strongest feature importances include days with impressions, log impressions, average position and content age. Feature importance is associative and does not prove causation.

## Self-check
- [x] Multiple models compared
- [x] Same split and metric
- [x] Errors/interpretation framed carefully

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance

# 1. Resolve repository root
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'scripts' / '02_baseline_score.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# 2. Load dataset
data_path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

df = pd.read_csv(data_path)

# 3. Dynamic Feature Mapping
col_ctr = next((c for c in ['ctr', 'ctr_vs_position', 'ctr_gap'] if c in df.columns), 'ctr')
col_staleness = next((c for c in ['days_since_last_update', 'staleness_days', 'staleness'] if c in df.columns), 'days_since_last_update')
col_pos = next((c for c in ['avg_position', 'position', 'pos'] if c in df.columns), 'avg_position')
col_imp = next((c for c in ['impressions_90d', 'impressions_30d', 'impressions'] if c in df.columns), 'impressions')

# Target setup
col_target = next((c for c in ['is_declining', 'declining', 'target', 'decline_flag'] if c in df.columns), None)
if col_target is None or col_target not in df.columns:
    df['is_declining'] = (df[col_ctr] < 0.05).astype(int)
    col_target = 'is_declining'
else:
    df[col_target] = df[col_target].astype(int)

features = [col_ctr, col_staleness, col_pos, col_imp]
X = df[features].fillna(df[features].median())
y = df[col_target]

# ------------------------------------------------------------------
# SECTION 1 & 2: METHOD CHOICE & VALIDATION SPLIT DESIGN
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("=" * 65)
print("SECTION 1 & 2: TRAIN-TEST SPLIT SUMMARY")
print("=" * 65)
print(f"Train Set: {X_train.shape[0]} rows | Test Set: {X_test.shape[0]} rows")
print(f"Target Distribution (Train): {y_train.mean():.4f} | (Test): {y_test.mean():.4f}\n")

# ------------------------------------------------------------------
# SECTION 3: MODEL TRAINING & BASELINE COMPARISON
# ------------------------------------------------------------------
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

baseline_score_test = (
    (1 - X_test[col_ctr].rank(pct=True)) * 0.40 +
    X_test[col_staleness].rank(pct=True) * 0.30 +
    X_test[col_pos].rank(pct=True) * 0.25 +
    X_test[col_imp].rank(pct=True) * 0.05
)
y_pred_baseline = (baseline_score_test > baseline_score_test.median()).astype(int)

metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'W04 Baseline Rule': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline, zero_division=0),
        recall_score(y_test, y_pred_baseline, zero_division=0),
        f1_score(y_test, y_pred_baseline, zero_division=0),
        roc_auc_score(y_test, baseline_score_test)
    ],
    'W05 Random Forest': [
        accuracy_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

print("=" * 65)
print("SECTION 3: MODEL VS BASELINE COMPARISON TABLE")
print("=" * 65)
print(metrics_summary.to_string(index=False))
print("\n")

# ------------------------------------------------------------------
# SECTION 4: PERMUTATION IMPORTANCE & ERROR ANALYSIS
# ------------------------------------------------------------------
print("=" * 65)
print("SECTION 4: FEATURE IMPORTANCE & ERROR ANALYSIS")
print("=" * 65)

perm_importance = permutation_importance(rf_model, X_test, y_test, n_repeats=10, random_state=42)
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance_Mean': perm_importance.importances_mean
}).sort_values(by='Importance_Mean', ascending=False)

print("Permutation Feature Importance:")
print(feat_imp.to_string(index=False))

SECTION 1 & 2: TRAIN-TEST SPLIT SUMMARY
Train Set: 24000 rows | Test Set: 6000 rows
Target Distribution (Train): 0.4696 | (Test): 0.4695

SECTION 3: MODEL VS BASELINE COMPARISON TABLE
   Metric  W04 Baseline Rule  W05 Random Forest
 Accuracy           0.705500                1.0
Precision           0.675000                1.0
   Recall           0.718850                1.0
 F1-Score           0.696235                1.0
  ROC-AUC           0.803937                1.0


SECTION 4: FEATURE IMPORTANCE & ERROR ANALYSIS
Permutation Feature Importance:
               Feature  Importance_Mean
                   ctr         0.498667
days_since_last_update         0.000000
          avg_position         0.000000
       impressions_90d         0.000000
